In [1]:
import requests
import h5py
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# SETTINGS
# ============================================================

API_KEY = "6c8ec3d465c0bcf623a06a2b4c43408c"

target_id = 738558
target_snap = 99

headers = {
    "api-key": API_KEY
}

# ============================================================
# 1. DOWNLOAD THE MAIN PROGENITOR BRANCH
# ============================================================

mpb_url = (
    f"https://www.tng-project.org/api/"
    f"TNG300-1/snapshots/{target_snap}/subhalos/{target_id}/sublink/mpb.hdf5"
)

response = requests.get(mpb_url, headers=headers)
response.raise_for_status()

with open("tree.hdf5", "wb") as f:
    f.write(response.content)

print("Downloaded main progenitor branch.")


# ============================================================
# 2. READ THE MAIN PROGENITOR BRANCH
# ============================================================

with h5py.File("tree.hdf5", "r") as hdf:
    snap_nums = hdf["SnapNum"][:]
    subfind_ids = hdf["SubfindID"][:]
    masses = hdf["SubhaloMass"][:]

# Create lookup:
# snapshot -> primary subhalo ID + mass
primary_lookup = {}

for snap, subhalo_id, mass in zip(
    snap_nums,
    subfind_ids,
    masses
):
    primary_lookup[int(snap)] = {
        "SubhaloID": int(subhalo_id),
        "Mass": float(mass)
    }

print(f"Main progenitor branch contains {len(primary_lookup)} snapshots.")


# ============================================================
# 3. GET THE SIMPLE TREE FOR THE TARGET
# ============================================================

simple_url = (
    f"https://www.tng-project.org/api/"
    f"TNG300-1/snapshots/{target_snap}/subhalos/"
    f"{target_id}/sublink/simple.json"
)

response = requests.get(simple_url, headers=headers)
response.raise_for_status()

simple_tree = response.json()

mergers = simple_tree.get("Mergers", [])

print(f"\nNumber of merger events: {len(mergers)}")


# ============================================================
# 4. FUNCTION TO GET SECONDARY MASS
# ============================================================

def get_subhalo_mass(snap, subhalo_id):
    """
    Get the total bound SubhaloMass for a subhalo
    at a specific snapshot.
    """

    url = (
        f"https://www.tng-project.org/api/"
        f"TNG300-1/snapshots/{snap}/subhalos/{subhalo_id}/"
    )

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    data = response.json()

    return data["mass"]


# ============================================================
# 5. CALCULATE MERGER MASS RATIOS
# ============================================================

merger_results = []

for snap, secondary_id in mergers:

    snap = int(snap)
    secondary_id = int(secondary_id)

    # --------------------------------------------------------
    # Primary
    # --------------------------------------------------------

    if snap not in primary_lookup:
        print(
            f"Skipping merger at snap {snap}: "
            f"primary not found in main progenitor branch."
        )
        continue

    primary_id = primary_lookup[snap]["SubhaloID"]
    primary_mass = primary_lookup[snap]["Mass"]

    # --------------------------------------------------------
    # Secondary
    # --------------------------------------------------------

    try:
        secondary_mass = get_subhalo_mass(
            snap,
            secondary_id
        )
    except Exception as e:
        print(
            f"Could not retrieve secondary "
            f"{secondary_id} at snap {snap}: {e}"
        )
        continue

    # --------------------------------------------------------
    # Mass ratio
    # --------------------------------------------------------

    if primary_mass > 0:
        mass_ratio = secondary_mass / primary_mass
    else:
        mass_ratio = np.nan

    # --------------------------------------------------------
    # Classification
    # --------------------------------------------------------

    if mass_ratio >= 0.25:
        merger_type = "Major"
    elif mass_ratio >= 0.10:
        merger_type = "Minor"
    else:
        merger_type = "Very minor"

    merger_results.append({
        "SnapNum": snap,
        "PrimarySubhaloID": primary_id,
        "SecondarySubhaloID": secondary_id,
        "PrimaryMass": primary_mass,
        "SecondaryMass": secondary_mass,
        "MassRatio": mass_ratio,
        "MergerType": merger_type
    })


# ============================================================
# 6. PUT EVERYTHING INTO A DATAFRAME
# ============================================================

merger_df = pd.DataFrame(merger_results)

if len(merger_df) > 0:

    merger_df = merger_df.sort_values(
        "SnapNum",
        ascending=False
    ).reset_index(drop=True)

    print("\n" + "=" * 80)
    print("MERGER HISTORY")
    print("=" * 80)

    print(
        merger_df.to_string(
            index=False,
            float_format=lambda x: f"{x:.4e}"
        )
    )

else:
    print("\nNo merger events were found.")


# ============================================================
# 7. SUMMARY
# ============================================================

if len(merger_df) > 0:

    print("\n" + "=" * 80)
    print("MERGER SUMMARY")
    print("=" * 80)

    print(
        f"Total mergers: "
        f"{len(merger_df)}"
    )

    print(
        f"Major mergers: "
        f"{(merger_df['MergerType'] == 'Major').sum()}"
    )

    print(
        f"Minor mergers: "
        f"{(merger_df['MergerType'] == 'Minor').sum()}"
    )

    print(
        f"Very minor mergers: "
        f"{(merger_df['MergerType'] == 'Very minor').sum()}"
    )

    print(
        f"Largest mass ratio: "
        f"{merger_df['MassRatio'].max():.4f}"
    )


# ============================================================
# 8. SAVE THE MERGER HISTORY
# ============================================================

merger_df.to_csv(
    "merger_history_738558.csv",
    index=False
)

print("\nSaved to merger_history_738558.csv")

Downloaded main progenitor branch.
Main progenitor branch contains 96 snapshots.

Number of merger events: 42

MERGER HISTORY
 SnapNum  PrimarySubhaloID  SecondarySubhaloID  PrimaryMass  SecondaryMass  MassRatio MergerType
      55            317979              318069   6.7493e+01     1.1085e-01 1.6423e-03 Very minor
      53            295530              295538   5.5714e+01     4.8852e+00 8.7684e-02 Very minor
      53            295530              295552   5.5714e+01     8.4813e-01 1.5223e-02 Very minor
      53            295530              295721   5.5714e+01     7.9669e-02 1.4300e-03 Very minor
      50            355694              355723   6.4058e+01     1.8640e-01 2.9099e-03 Very minor
      49            402296              402338   6.5751e+01     6.5387e-02 9.9446e-04 Very minor
      42            887568              887583   5.0149e+01     9.1619e-02 1.8269e-03 Very minor
      41            842182              842193   5.0990e+01     7.8624e-02 1.5420e-03 Very minor
 